# IMPORTS & FUNCTIONS

## IMPORTS

In [ ]:
import os
import re
import gc
import math
import copy
import json
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from tqdm.notebook import tqdm
from IPython.display import Video

from scipy.ndimage import center_of_mass

from skimage import io, img_as_float
from skimage.color import rgb2lab, lab2rgb
from skimage.filters import gaussian
from skimage.segmentation import slic, mark_boundaries, find_boundaries

import cv2
import imageio.v2 as imageio


## FUNCTIONS

In [ ]:
def list_files(folder_path):
    try:
        files = os.listdir(folder_path)
        valid_files = []
        
        for file in files:
            file_path = os.path.join(folder_path, file)
            if os.path.isfile(file_path): valid_files.append(file_path) 
        
        return valid_files 
    except Exception as e:
        return f"Errore: {e}"

## CLASS DEFINITIONS

The execution's pipeline is managed by a OP approach

### SEGMENTATION

In [ ]:
class Centroid:
    CURR_IDX = 0
    
    def __init__(self,img, y,x ):
        self.y = int(y)
        self.x = int(x)
        self.lab = img[self.y,self.x]

        self.idx = self.__class__.CURR_IDX
        self.__class__.CURR_IDX += 1
        self.error = None    

    @property
    def state(self): return np.hstack([self.y,self.x,self.lab.flatten()])
    
    def __repr__(self):
        state_str = " ".join(f"{val:10.5f}" for val in self.state)
        error_str = f"{self.error:10.5f}" if self.error is not None else " " * 10 + "None"
        return f"{self.idx:3}: {state_str} ; error: {error_str}"

    def move(self, img,y,x):
        y,x = int(y),int(x)
        old_state  = np.copy(self.state)
        self.state = np.hstack([y,x,img[y,x].flatten()])
        self.error = np.linalg.norm(self.state - old_state,2)

In [ ]:
class Superpixel:    
    #avoid escessive usage of ram
    def reset_info(self):
        self._mask = None
        self._mean_rgb = None
        self._patch = None

    def __init__(self, c, p):
        self.idx = c.idx
        self.pos = np.array([c.y, c.x])
        self._p = p  # image
        self._mask = None
        self._mean_rgb = None
        self._patch = None

    @property
    def y(self): return int(self.pos[0])
    @property
    def x(self): return int(self.pos[1])

##LAZY EVALUATION YO 
    @property
    def mask(self):
        if self._mask is None:
            self._mask =  (self._p.owner_map == self.idx)
        return self._mask

    @property
    def mean_rgb(self):
        if self._mean_rgb is None:
            self._mean_rgb = np.mean(self._p.original[self.mask], axis=0)
        return self._mean_rgb

    @property
    def patch(self):
        if self._patch is None:
            self._patch = self.__class__.patch_descriptor(self._p.original, self.y, self.x)
        return self._patch
    


    #HOW TO DESCRIBE
    def patch_descriptor(I, y, x, size_w=11):
        hw = size_w // 2  # Half-width
        I_ext = np.pad(I, hw, mode='reflect')  # Pad image to handle edges

        y_p = y + hw
        x_p = x + hw

        patch = I_ext[y_p - hw:y_p + hw + 1, x_p - hw:x_p + hw + 1]
        return patch.flatten()

In [ ]:
class BaseSLIC: #superclass
    def __init__(self, img, num_segments=100, M=10,smooth=0):
        self.original = np.copy(img / 255.0 if img.dtype == np.uint8 or img.max() > 1.0 else img)
        self.data = rgb2lab(self.original if smooth is None or smooth <= 0 else gaussian(self.original, sigma=smooth,channel_axis=-1),channel_axis=-1)

        self.H, self.W = self.original.shape[:2]
        self.N = self.H * self.W

        self.K = num_segments
        self.M = M
        self.S = int(math.sqrt(self.N / self.K))

    def show_boundaries(self,
        img="original",
        centroids=False,
        c_idx=False,
        bound_color=(1, 1, 0.1),
        c_color = (0.3,1,0.3),
        legend=True,
        ax=None
        ):
        if ax is None:
            fig = plt.figure(f"Superpixels -- {self.K} segments")
            ax = fig.add_subplot(1, 1, 1)

        c_text = (1,0.3,0.3)
        

        base = self.original if img == "original" else (lab2rgb(self.data) if img=="data" else np.zeros_like(self.original))
        
        ax.imshow(mark_boundaries(base, self.owner_map, color=bound_color))
        ax.axis("off")

        if centroids or c_idx:
            for c in self.centroids:
                if centroids:
                    ax.plot(c.x, c.y, 'o', color=c_color, markersize=3)
                if c_idx:
                    ax.text(c.x+3, c.y, c.idx, color=c_text, ha='center')

        if legend:
            ax.set_title("Superpixels Segmentation (SLIC)")
            legend_elements = [
                Patch(facecolor=c_color, edgecolor='black', label='Centroids') if centroids else None,
                Patch(facecolor=c_text, edgecolor='black', label='Indices') if c_idx else None,
                Patch(facecolor=bound_color, edgecolor='black', label='Boundary'),
            ]
            ax.legend(
                handles=[elem for elem in legend_elements if elem is not None], #remove nulls
                loc='upper right',
                frameon=True,              # Show the legend frame
                facecolor='white',         # Set background color
                edgecolor='black'          # Optional: frame edge color
            )

    def explained_color_variance(self, ax=None):
        image = self.original
        labels = self.owner_map

        total_variance = 0.0
        all_variances = []

        for label in np.unique(labels):
            mask = labels == label
            pixels = image[mask]  # (N, 3)

            if len(pixels) <= 1:
                continue

            var = np.var(pixels, axis=0)
            segment_variance = np.sum(var)  # Sum over R/G/B or L/a/b
            all_variances.append(segment_variance)
            total_variance += segment_variance

        total_variance /= len(self.centroids)
        # Plot histogram with normalized y-values (relative to number of segments)
        if ax is None:
            fig, ax = plt.subplots(figsize=(6, 4))

        ax.hist(all_variances, bins=30, color='skyblue', edgecolor='black', density=True)
        ax.set_title(f"Mean Color Variance: {total_variance:.4f}")
        ax.set_xlabel("SuperPixel's internal variance")
        ax.set_ylabel("Density")
        ax.set_yticks([])

        return total_variance


In [ ]:
class SLIC_old(BaseSLIC):
    NORM_XY = 2
    NORM_LAB = 2
    distance_masks = {}

    @classmethod
    def init_distance_mask(cls, M,S):
        distance_mask = np.empty((2*S+1, 2*S+1), dtype=float)
        for dy in range(-S, S+1):
            for dx in range(-S, S+1):
                dist = np.linalg.norm([dy, dx], cls.NORM_XY)
                distance_mask[dy+S, dx+S] = dist     
        return distance_mask * (M / S)

    @classmethod
    def get_mask(cls, M,S):
        if S not in cls.distance_masks:
            cls.distance_masks[(M,S)] = cls.init_distance_mask(M,S)
        return cls.distance_masks[(M,S)]

    def __init__(self, img, num_segments=100, M=10, max_iter=10, error_threshold=0, verbose=False, smooth=None):
        super().__init__(img, num_segments, M,smooth=smooth)
        self.T = max_iter
        self.error_threshold = error_threshold
        self.verbose = verbose

        self.centroids = []
        self.owner_map = np.full((self.H, self.W), -1, dtype=int)
        self.dist_map = np.full((self.H, self.W), np.inf, dtype=float)
        self.errors = []

        self.process()
        self.SP_list = np.array([Superpixel(c,self) for c in self.centroids])

    def create_centroids(self):
        Centroid.CURR_IDX = 0
        self.centroids = []
        y = self.S // 2
        while y < self.H:
            x = self.S // 2
            while x < self.W:
                self.centroids.append(Centroid(self.data, y, x))
                x += self.S
            y += self.S
        self.K = len(self.centroids)

    def gradient(self, y, x):
        y, x = int(y), int(x)
        if x + 1 > self.W - 1: x = self.W - 2
        if y + 1 > self.H - 1: y = self.H - 2
        if x == 0: x = 1
        if y == 0: y = 1
        g_x = np.linalg.norm(self.data[y, x + 1] - self.data[y, x - 1], 2)
        g_y = np.linalg.norm(self.data[y + 1, x] - self.data[y - 1, x], 2)
        return g_x + g_y

    def __align_centroids(self, n_size=3):
        if n_size % 2 == 0: print("neighbour_size needs to be AT LEAST odd")
        for c in self.centroids:
            min_grad = float('inf')
            min_pos_x, min_pos_y = None, None
            for dx in range(-n_size//2, n_size//2 + 1):
                for dy in range(-n_size//2, n_size//2 + 1):
                    pos_x, pos_y = c.x + dx, c.y + dy
                    if not (0 <= pos_x < self.W and 0 <= pos_y < self.H): continue
                    curr_grad = self.gradient(pos_y, pos_x)
                    if curr_grad < min_grad:
                        min_grad = curr_grad
                        min_pos_x, min_pos_y = pos_x, pos_y
            c.move(self.data, min_pos_y, min_pos_x)

    #optimization by chatgpt; vectorialize readings and writings
    def __assignment(self):
        distance_mask = self.__class__.get_mask(self.M,self.S)
        for c in self.centroids:
            c_y, c_x, c_lab, c_idx = c.y, c.x, c.get_lab(), c.idx
            y_start = max(c_y - self.S, 0)
            y_end = min(c_y + self.S + 1, self.H)
            x_start = max(c_x - self.S, 0)
            x_end = min(c_x + self.S + 1, self.W)
            window = self.data[y_start:y_end, x_start:x_end]
            h, w = window.shape[:2]
            mask_y_start = y_start - (c_y - self.S)
            mask_y_end = mask_y_start + h
            mask_x_start = x_start - (c_x - self.S)
            mask_x_end = mask_x_start + w
            D_xy = distance_mask[mask_y_start:mask_y_end, mask_x_start:mask_x_end]
            diff = window - c_lab
            D_lab = np.linalg.norm(diff, axis=2, ord=self.__class__.NORM_LAB)
            D = D_lab + D_xy
            current_D = self.dist_map[y_start:y_end, x_start:x_end]
            current_O = self.owner_map[y_start:y_end, x_start:x_end]
            mask = D < current_D
            current_D[mask] = D[mask]
            current_O[mask] = c_idx

    def __update_cluster(self):
        for c in self.centroids:
            y_vals, x_vals = np.where(self.owner_map == c.idx)
            if len(x_vals) == 0: continue
            mean_x = int(np.mean(x_vals))
            mean_y = int(np.mean(y_vals))
            c.move(self.data, mean_y, mean_x)

    def get_residual_error(self):
        return sum(c.error for c in self.centroids)

    def process(self):
        if not self.centroids:
            self.create_centroids()
            self.__align_centroids()
        for i in tqdm(range(self.T), desc="SLIC iteration", disable=not self.verbose, leave=None):
            self.__assignment()
            self.__update_cluster()
            tot_error = self.get_residual_error()
            self.errors.append(tot_error)
            if tot_error <= self.error_threshold:
                return True
        return False

In [ ]:
class SLIC(BaseSLIC):
    def __init__(self, img, num_segments=100, M=10, max_iter=10, error_threshold=0, verbose=True, smooth=None):
        super().__init__(img, num_segments, M,smooth=smooth)
        self.T = max_iter
        self.error_threshold = error_threshold
        self.verbose = verbose
        self.errors = []

        self.owner_map = slic(self.data, n_segments=self.K, compactness=self.M, start_label=0,enforce_connectivity=False)
        self.centroids = []

        unique_labels = np.unique(self.owner_map)
        centers = center_of_mass(np.ones_like(self.owner_map), labels=self.owner_map, index=unique_labels)
        Centroid.CURR_IDX = 0
        for y, x in centers:
            self.centroids.append(Centroid(self.data, y, x))
            
        self.SP_list = np.array([Superpixel(c,self) for c in self.centroids])

### MATCHING

In [ ]:
class DISTANCE:
    @staticmethod
    def iou(a: Superpixel, b: Superpixel, **params):
        i = np.sum(a.mask & b.mask)
        u = np.sum(a.mask | b.mask)
        return -i/u #minus to make that bigger==movement
    
    @staticmethod
    def overlap(a: Superpixel, b: Superpixel, **params):
        a_only = np.sum(a.mask & ~b.mask) / np.sum(a.mask)
        b_only = np.sum(b.mask & ~a.mask) / np.sum(b.mask)
        return min(a_only,b_only) * 100 #percentage
    
    @staticmethod
    def xy(a: Superpixel, b: Superpixel, **params):
        dist = np.linalg.norm(a.pos - b.pos, 2)
        return dist
    
    @staticmethod
    def rgb(a: Superpixel, b: Superpixel, **params):
        dist = np.linalg.norm((a.mean_rgb-b.mean_rgb), 2)
        return dist
    
    @staticmethod
    def patch(a: Superpixel, b: Superpixel, **params):
        dist = np.linalg.norm((a.patch-b.patch), 2)
        return dist

In [ ]:
class KERNEL:
    @staticmethod
    def xy(a: Superpixel, b: Superpixel,s_xy=1, **params):   
        dist = DISTANCE.xy(a,b,**params)**2
        return np.exp(-dist/(2*s_xy**2))
    
    @staticmethod
    def rgb(a: Superpixel, b: Superpixel, s_rgb=1,**params):
        dist = DISTANCE.rgb(a,b,**params)**2
        return np.exp(-dist/(2*s_rgb**2))
    
    @staticmethod
    def rgb_xy(a: Superpixel, b: Superpixel, alpha=0.5,**params):
        xy = KERNEL.xy(a,b,**params)
        rgb = KERNEL.rgb(a,b,**params)
        return (alpha * rgb + (1-alpha) * xy)

    @staticmethod
    def patch(a: Superpixel, b: Superpixel, s_patch=1,**params):
        dist = DISTANCE.patch(a,b,**params)**2
        return np.exp(-dist/(2*s_patch**2))
    
    @staticmethod
    def patch_xy(a: Superpixel, b: Superpixel, alpha=0.5,**params):
        xy = KERNEL.xy(a,b,**params)
        patch = KERNEL.patch(a,b,**params)
        return (alpha * patch + (1-alpha) * xy)

In [ ]:
class Matcher:
    def __init__(self, prev, curr, **params):
        self.prev = prev
        self.curr = curr
        
        self.SP_prev = prev.SP_list
        self.SP_curr = curr.SP_list

        self.choices_prev = np.empty_like(self.SP_prev)
        self.choices_curr = np.empty_like(self.SP_curr)
        self.choices_both = []

        self.kernel_name = params['kernel'].__name__
        self.similarity_matrix = None
        self.match(**params)

    def get_current_SP(self,idx):
        for SP in self.SP_curr:
            if SP.idx == idx:
                return SP
        raise Exception(f"SP with idx={idx} does not exist")

        
    def create_matrix(self,**params):
        kernel = params['kernel']
        self.similarity_matrix = np.empty((len(self.SP_prev),len(self.SP_curr)))
        for i, a in enumerate(self.SP_prev):
            for j, b in enumerate(self.SP_curr):
                self.similarity_matrix[i,j] = kernel(a,b,**params)
    
    def match(self,**params):
        self.create_matrix(**params)
        choices_prev = np.argmax(self.similarity_matrix, axis=1)  # best b for each a
        choices_curr = np.argmax(self.similarity_matrix, axis=0)  # best a for each b
        
        for i, j in enumerate(choices_prev):
            if choices_curr[j] == i: #reciprocal 
                self.choices_both.append((self.SP_prev[i], self.SP_curr[j]))

        self.choices_prev = [(self.SP_prev[i],self.SP_curr[j]) for i,j in enumerate(choices_prev)]
        self.choices_curr = [(self.SP_prev[j],self.SP_curr[i]) for i,j in enumerate(choices_curr)]
    
############# PLOTS #################
    
    def show_overlay(self, ax=None, centroids=False, img_bg=None):
        self.__class__.plot_overlay(prev=self.prev, curr=self.curr, ax=ax, centroids=centroids,img_bg = img_bg)
    
    @staticmethod
    def plot_overlay(prev, curr, ax=None, centroids=False,img_bg = None):
        if ax is None:
            fig, ax = plt.subplots(figsize=(8, 8))

        prev_c = (0.8, 0, 0)
        curr_c = (0, 0.8, 0)

        img_base = img_bg if img_bg is not None else np.full_like(prev.original, (128, 128, 128))  # white background
        img_with_boundaries = mark_boundaries(img_base, prev.owner_map, color=prev_c) 

        # Plot the base with prev boundaries
        img_with_both = mark_boundaries(img_with_boundaries, curr.owner_map, color=curr_c)  # green for curr

        ax.imshow(img_with_both)
        ax.axis("off")
        ax.set_title("Superpixels Overlay")

        if centroids:
            # Plot centroids on top
            for c in prev.centroids:
                ax.plot(c.x, c.y, '.', color=prev_c, markersize=5)  # white for prev
            for c in curr.centroids:
                ax.plot(c.x, c.y, '.', color=curr_c, markersize=5)  # black for curr

        if ax is None:
            plt.show()

        legend_elements = [
            Patch(facecolor=prev_c, edgecolor='black', label='Previous'),
            Patch(facecolor=curr_c, edgecolor='black', label='Current')
        ]
        ax.legend(
            handles=legend_elements,
            loc='upper right',
            frameon=True,              # Show the legend frame
            facecolor='white',         # Set background color
            edgecolor='black'          # Optional: frame edge color
        )

    def plot_matrix(self, ax=None):
        if ax is None:
            fig, ax = plt.subplots(figsize=(8, 6))

        im = ax.imshow(self.similarity_matrix, cmap='viridis')
        cbar = plt.colorbar(im, ax=ax, label="Similarity")
        
        ax.set_title(f"Similarity Matrix of \nKERNEL.{self.kernel_name}(a,b)")
        ax.set_xlabel("Superpixels t+1")
        ax.set_ylabel("Superpixels t")
        plt.tight_layout()

        if ax is None:
            plt.show()

    def show_match(self, type="both",ax=None, boundaries=False,tresh_xy=0,idx_pair=None,c_unmatched = (0.9,0.9, 0.9)):
        """Show matches between previous and current superpixels using centroid positions and optional boundaries."""

        img1 = self.prev.original.copy()
        img2 = self.curr.original.copy()

        if ax is None:
            fig, ax = plt.subplots(figsize=(15, 10))


        # Centroids
        corners1 = np.array([sp.pos for sp in self.SP_prev])
        corners2 = np.array([sp.pos for sp in self.SP_curr])

        #chooses which pairs of SP to use
        pairs = {
            "both": self.choices_both,
            "prev": self.choices_prev,
            "curr": self.choices_curr
        }.get(type)        
        if pairs is None: raise Exception("Type of the pairs can only be 'both','prev' or 'curr'")

        prev_c = (0.8, 0, 0); prev_centroid_c = (0.3,0,0)
        curr_c = (0, 0.8, 0); curr_centroid_c = (0,0.3,0)
        line_c = (0, 0.75 , 0.75)

        # Create a list of matched superpixels
        matched_prev = np.array([a.idx for a, _ in pairs])
        for sp in self.SP_prev: 
            if sp.idx not in matched_prev: 
                img1[sp.mask] = c_unmatched

        matched_curr = np.array([b.idx for _, b in pairs])
        for sp in self.SP_curr:
            if sp.idx not in matched_curr:
                img2[sp.mask] = c_unmatched


                # Add boundaries if requested
        if boundaries:
            img1 = mark_boundaries(img1, self.prev.owner_map, color=prev_c)  # green for prev
            img2 = mark_boundaries(img2, self.curr.owner_map, color=curr_c)  # red for curr


        ax.imshow(np.concatenate([img1, img2], axis=1))
        ax.scatter(corners1[:, 1], corners1[:, 0], s=20, color=prev_centroid_c,marker="x")
        ax.scatter(corners2[:, 1] + img1.shape[1], corners2[:, 0], s=20,color= curr_centroid_c,marker="x")


        #show only 1, if requested
        if (idx_pair is not None):
            # Ensure idx_pair is always iterable
            idx_pair = [idx_pair] if isinstance(idx_pair, (int, float)) else idx_pair
            for idx in idx_pair:
                # Iterate over all pairs to find the one with a.idx == idx
                for a, b in pairs:
                    if (a.idx == idx and type!="curr") | (b.idx == idx and type=="curr"):
                        (y1, x1), (y2, x2) = a.pos, b.pos
                        ax.plot([x1, x2 + img1.shape[1]], [y1, y2], color=line_c)
                        ax.scatter([x1,x2+ img1.shape[1]], [y1,y2], s=20, color=line_c,marker="o")
        else:
            matches = [(a.pos, b.pos) for a, b in pairs if DISTANCE.xy(a, b) >= tresh_xy]

            for (y1, x1), (y2, x2) in matches:
                ax.plot([x1, x2 + img1.shape[1]], [y1, y2], color=line_c)
                ax.scatter([x1,x2+ img1.shape[1]], [y1,y2], s=20, color=line_c,marker="o")

        ax.axis("off")
        ax.set_title("Superpixel Matches" + (" with Boundaries" if boundaries else ""))


        legend_elements = [
            Patch(facecolor=c_unmatched, edgecolor='black', label='Unmatched SPs'),
            Patch(facecolor=line_c, edgecolor='black', label='Connection'),
            Patch(facecolor=prev_c, edgecolor='black', label='Previous SPs'),
            Patch(facecolor=curr_c, edgecolor='black', label='Current SPs')
        ]
        ax.legend(
            handles=legend_elements,
            loc='upper right',
            frameon=True,              # Show the legend frame
            facecolor='white',         # Set background color
            edgecolor='black'          # Optional: frame edge color
        )

    def show_choices(self, type="prev", ax=None, boundaries=False, text_color=(0, 0, 0), text_bg=(1,1,1),text_size={"start":4,"step":4}):
        img = (self.prev.original.copy() if type == "prev" else self.curr.original.copy())

        choices = {
            "prev": self.choices_curr,
            "curr": self.choices_prev
        }.get(type)
        if choices is None:
            raise Exception("Type must be 'prev' or 'curr'.")

        count_map = {}
        for a, b in choices:
            key = a.idx if type == "prev" else b.idx
            count_map[key] = count_map.get(key, 0) + 1

        if boundaries:
            owner_map = self.prev.owner_map if type == "prev" else self.curr.owner_map
            img = mark_boundaries(img, owner_map, color=boundaries)

        if ax is None:
            fig, ax = plt.subplots(figsize=(8, 8))

        ax.imshow(img)
        ax.axis("off")
        
        SP_list = self.SP_prev if type == "prev" else self.SP_curr

        for sp in SP_list:
            count = count_map.get(sp.idx, 0)
            if count > 0:
                # Apply log scale to size (adjust scale factor as needed)
                log_count = np.log(count + 1)  # log(count) with log(0) = 0, so adding 1
                fontsize = text_size["start"] + log_count * text_size["step"]  # Adjust multiplier to control the size scaling

                ax.text(
                    sp.x, sp.y,
                    str(count),
                    color=text_color,
                    fontsize=fontsize,  # Dynamic fontsize based on log scale
                    ha='center',
                    va='center',
                    bbox=dict(facecolor=text_bg, alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2')
                )

        ax.set_title(f"Counts of matching of Superpixels in {type.capitalize()}")

    def show_match_v2(self, type="both", ax=None, boundaries=False, tresh_xy=0, idx_pair=None, c_unmatched=(0.9, 0.9, 0.9), alpha=0.5,legend=True):

        img1 = self.prev.original.copy()
        img2 = self.curr.original.copy()

        if ax is None:
            fig, ax = plt.subplots(figsize=(15, 10))

        # Centroids
        corners1 = np.array([sp.pos for sp in self.SP_prev])
        corners2 = np.array([sp.pos for sp in self.SP_curr])

        # Choose which pairs of SP to use
        pairs = {
            "both": self.choices_both,
            "prev": self.choices_prev,
            "curr": self.choices_curr
        }.get(type)
        if pairs is None:
            raise Exception("Type of the pairs can only be 'both', 'prev', or 'curr'")

        prev_c = (0.8, 0, 0)
        prev_centroid_c = (0.3, 0, 0)
        curr_c = (0, 0.8, 0)
        curr_centroid_c = (0, 0.3, 0)
        line_c = (0, 0.75, 0.75)

        # Create a list of matched superpixels
        matched_prev = np.array([a.idx for a, _ in pairs])
        for sp in self.SP_prev:
            if sp.idx not in matched_prev:
                img1[sp.mask] = c_unmatched

        matched_curr = np.array([b.idx for _, b in pairs])
        for sp in self.SP_curr:
            if sp.idx not in matched_curr:
                img2[sp.mask] = c_unmatched

        # Add boundaries if requested
        if boundaries:
            img1 = mark_boundaries(img1, self.prev.owner_map, color=prev_c)  # green for prev
            img2 = mark_boundaries(img2, self.curr.owner_map, color=curr_c)  # red for curr

        # Overlap the images with alpha blending
        overlapped_img = cv2.addWeighted(img1, alpha, img2, 1 - alpha, 0)

        ax.imshow(overlapped_img)
        ax.scatter(corners1[:, 1], corners1[:, 0], s=20, color=prev_centroid_c, marker="x")
        ax.scatter(corners2[:, 1], corners2[:, 0], s=20, color=curr_centroid_c, marker="x")

        # Show only one pair if requested
        if idx_pair is not None:
            # Ensure idx_pair is always iterable
            idx_pair = [idx_pair] if isinstance(idx_pair, (int, float)) else idx_pair
            for idx in idx_pair:
                # Iterate over all pairs to find the one with a.idx == idx
                for a, b in pairs:
                    if (a.idx == idx and type != "curr") or (b.idx == idx and type == "curr"):
                        (y1, x1), (y2, x2) = a.pos, b.pos
                        ax.plot([x1, x2], [y1, y2], color=line_c)
                        ax.scatter([x1, x2], [y1, y2], s=20, color=line_c, marker="o")
        else:
            matches = [(a.pos, b.pos) for a, b in pairs if DISTANCE.xy(a, b) >= tresh_xy]

            for (y1, x1), (y2, x2) in matches:
                ax.plot([x1, x2], [y1, y2], color=line_c)
                ax.scatter([x1, x2], [y1, y2], s=20, color=line_c, marker="o")

        ax.axis("off")
        ax.set_title("Superpixel Matches" + (" with Boundaries" if boundaries else ""))

        if legend:
            legend_elements = [
                Patch(facecolor=c_unmatched, edgecolor='black', label='Unmatched SPs'),
                Patch(facecolor=line_c, edgecolor='black', label='Displacement'),
                Patch(facecolor=prev_c, edgecolor='black', label='Previous SPs'),
                Patch(facecolor=curr_c, edgecolor='black', label='Current SPs')
            ]
            ax.legend(
                handles=legend_elements,
                loc='upper right',
                frameon=True,
                facecolor='white',
                edgecolor='black'
            )



### DETECTION

In [ ]:

class Detector:
    def __init__(self,matcher,**params):
        self.matcher = matcher
        self.tresh = params['tresh']
        self.dist_name = params['dist'].__name__
        self.detect(**params)

    def detect(self,dist,tresh,type="both",**params):
        self.prev_moved_ids = []
        self.curr_moved_ids = []

        self.prev_moved_mask = None
        self.curr_moved_mask = None

        #chooses which pairs of SP to use
        pairs = {
            "both": self.matcher.choices_both,
            "prev": self.matcher.choices_prev,
            "curr": self.matcher.choices_curr
        }.get(type)
        
        if pairs is None: raise Exception("Type of the pairs can only be 'both','prev' or 'curr'")

        # Initialize masks
        self.prev_moved_mask = np.zeros_like(self.matcher.prev.owner_map, dtype=bool)
        self.curr_moved_mask = np.zeros_like(self.matcher.curr.owner_map, dtype=bool)

        for sp_prev,sp_curr in pairs:
            #selct only some superpixels
            distance = dist(sp_prev,sp_curr)
            good = (distance >= tresh)

            self.prev_moved_ids.append((sp_prev.idx,distance,good))
            self.curr_moved_ids.append((sp_curr.idx,distance,good))
            if good :                
                #adding to the masks the superpixel's region
                self.prev_moved_mask |= self.matcher.prev.owner_map == sp_prev.idx
                self.curr_moved_mask |= self.matcher.curr.owner_map == sp_curr.idx

            sp_prev.reset_info()




    ##PLOTS ##


    def show_mask(self,ax=None,c_background=(0.6,0.6,0.6),consider_unmatched=True,legend=True):
        image = self.matcher.curr.original.copy()
        moved_mask = self.curr_moved_mask.copy()
        background = np.full_like(image,c_background)  # arancione RGB


        if consider_unmatched:
            matched_curr = np.array([b.idx for _, b in self.matcher.choices_both])
            for sp in self.matcher.SP_curr:
                if sp.idx not in matched_curr:
                    moved_mask |=  self.matcher.curr.owner_map == sp.idx



        background[moved_mask] = image[moved_mask]
        
        if ax is None:
            fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(background)
        if legend:ax.set_title("Detected movement")
        ax.set_axis_off()

    
    
    def show_overlay(
            self, 
            ax=None, 
            alpha=0.5, 
            unmatched_color=(1, 0.3, 0.3), 
            boundary_color=(1,1,1), 
            moved_mask_color=(0.2,1,0.2),
            text_color=(1,1,1),
            show_dist=False,
            legend=True
            ):
        # Start from original image
        img = self.matcher.curr.original.copy()

        # Blend boundary map from owner_map
        boundaries = find_boundaries(self.matcher.curr.owner_map)
        img[boundaries] = (
            alpha * np.array(boundary_color) + (1 - alpha) * img[boundaries]
        )

        exists_unmatched = False
        # Color unmatched patches
        matched_curr = np.array([b.idx for _, b in self.matcher.choices_both])
        for sp in self.matcher.SP_curr:
            if sp.idx not in matched_curr:
                exists_unmatched = True
                img[sp.mask] = unmatched_color

        # Blend moved mask directly (not boundaries)
        mask = self.curr_moved_mask
        img[mask] = (alpha * np.array(moved_mask_color) + (1 - alpha) * img[mask])
        
        
        # Plot on provided axis or create new figure
        if ax is None:
            fig, ax = plt.subplots(figsize=(8, 8))


        if show_dist:
            for idx,dist,_ in self.curr_moved_ids:
                SP =self.matcher.get_current_SP(idx)
                ax.text(SP.x, SP.y, f"{dist:.1f}", color=text_color, fontsize=8, ha='center', va='center')
        
        ax.imshow(img)
        ax.axis("off")
        if legend:
            ax.set_title(f"Detector Visualization \nDISTANCE.{self.dist_name}(SP) > {self.tresh}")

            legend_elements = [
                Patch(facecolor=text_color, edgecolor='black', label=f'{self.dist_name}(SP)') if show_dist else None,
                Patch(facecolor=boundary_color, edgecolor='black', label='Boundary'),
                Patch(facecolor=unmatched_color, edgecolor='black', label='Unmatched SPs') if exists_unmatched else None,
                Patch(facecolor=moved_mask_color, edgecolor='black', label='Detected SPs')
            ]
            ax.legend(
                handles=[elem for elem in legend_elements if elem is not None],
                loc='upper right',
                frameon=True,              # Show the legend frame
                facecolor='white',         # Set background color
                edgecolor='black'          # Optional: frame edge color
            )

    def plot_distances_hist(self, bins=50, ax=None):
        distances = [d for _, d, _ in self.curr_moved_ids]

        if ax is None:
            fig, ax = plt.subplots(figsize=(6, 4))

        ax.hist(distances, bins=bins, color="skyblue", edgecolor="black")
        ax.set_xlabel(f"DISTANCE.{self.dist_name}(SP)")
        ax.set_ylabel("Frequency")
        ax.set_title("Histogram of SP's changes")
        ax.grid(True)
        ax.axvline(self.tresh, color='red', linestyle='--', label=f'Threshold = {self.tresh:.2f}')
        ax.legend()

###  UTILITY

In [ ]:
class FrameCollection:    
    @staticmethod
    def get_appropriate_class(detector_params):
        remember_rate = detector_params.get("remember_rate", 0)
        if remember_rate > 0:
            from __main__ import MemoryFrameCollection
            return MemoryFrameCollection
        return FrameCollection

    @staticmethod
    def create(frames_bank, subject, slic_params, matcher_params, detector_params):
        cls = FrameCollection.get_appropriate_class(detector_params)
        return cls(frames_bank, subject, slic_params, matcher_params, detector_params)


    def copy(self, slic_params=None, matcher_params=None, detector_params=None):

        # Deepcopy fallback parameters
        matcher_params = copy.deepcopy(matcher_params) if matcher_params is not None else self.matcher_params
        detector_params = copy.deepcopy(detector_params) if detector_params is not None else self.detector_params

        # CASE 1 — Recompute everything from scratch
        if slic_params is not None:
            new = FrameCollection.create(
                self.frames_bank,
                self.subject,
                slic_params,
                matcher_params,
                detector_params
            )

            if new.check_already_exists() is None:
                print("Computing SLIC")
                new.compute_all_frames()
            return new

        print("Segmentation unchanged")

        # CASE 2 — Reuse SLIC, change matching
        if matcher_params != self.matcher_params:
            new = FrameCollection.create(
                self.frames_bank,
                self.subject,
                self.slic_params,
                matcher_params,
                detector_params
            )

            if new.check_already_exists() is not None:
                return new

            if len(self.next_frames) > 0:
                print("Computing Frames of the original Collection")
                self.compute_all_frames()

            new.prev_frame = self.prev_frame
            new.next_frames = []

            for detector in tqdm(self.computed, desc="Recomputing Matchers"):
                slic_prev = detector.matcher.prev
                slic_curr = detector.matcher.curr

                new_matcher = Matcher(slic_prev, slic_curr, **matcher_params)
                new_detector = Detector(new_matcher, **detector_params)

                for SP in detector.matcher.SP_prev:
                    SP.reset_info()

                new.computed.append(new_detector)

            return new

        print("Matching unchanged")

        # CASE 3 — Reuse SLIC and matching, change detection
        if detector_params != self.detector_params:
            new = FrameCollection.create(
                self.frames_bank,
                self.subject,
                self.slic_params,
                self.matcher_params,
                detector_params
            )

            if new.check_already_exists() is not None:
                return new

            if len(self.next_frames) > 0:
                print("Computing Frames of the original Collection")
                self.compute_all_frames()

            new.prev_frame = self.prev_frame
            new.next_frames = []
            

            new.copy_detectors(self,**detector_params)

            return new

        print("Detection unchanged (Collection unchanged)")

        return copy.deepcopy(self)
    
    def copy_detectors(self,old,**detector_params):
        for detector in tqdm(old.computed, desc="Recomputing Detectors"):
                new_detector = Detector(detector.matcher, **detector_params)

                for SP in detector.matcher.SP_prev:
                    SP.reset_info()

                self.computed.append(new_detector)

    def __init__(self, frames_paths, subject="default",slic_params=None,matcher_params=None,detector_params=None):
        self.subject = subject
        self.computed = []
        self.frames_bank = frames_paths
        self.next_frames = frames_paths
        self.prev_frame = self.__get_new_frame()

        #eventually initialize them
        self.slic_params = slic_params
        self.matcher_params = matcher_params
        self.detector_params = detector_params

    ### EXECUTION ###    
    @classmethod
    def load_image(cls, path):
        img = io.imread(path)
        img = img_as_float(img)
        return img

    def __get_new_frame(self):
        if len(self.next_frames) == 0:
            raise Exception("No frames left to compute")
        img = self.__class__.load_image(self.next_frames[0])
        self.next_frames = self.next_frames[1:]
        return img

    def compute_frame(self):
        curr_frame = self.__get_new_frame()
        slic_prev = SLIC(self.prev_frame, **self.slic_params) if not self.computed else self.computed[-1].matcher.curr
        slic_curr = SLIC(curr_frame, **self.slic_params)
        matcher = Matcher(slic_prev, slic_curr, **self.matcher_params)
        detector = Detector(matcher, **self.detector_params)

        for SP in detector.matcher.SP_prev: SP.reset_info()

        self.computed.append(detector)
        self.prev_frame = curr_frame
        return detector

    def compute_all_frames(self):
        if len(self.next_frames) == 0:return 
        for _ in tqdm(range(len(self.next_frames)), desc="Processing Frames"):
            self.compute_frame()

    #### FILES ####

    @property
    def directory(self):
        return os.path.join("video_exports/basic/", self.subject)

    @property
    def metadata_file(self):
        return os.path.join(self.directory, "metadata.json")

    #### FILE METADATA #########
    def _load_metadata(self):
        if os.path.exists(self.metadata_file):
            with open(self.metadata_file, 'r') as f:
                return json.load(f)
        return []

    def _save_metadata(self, metadata):
        with open(self.metadata_file, 'w') as f:
            json.dump(metadata, f, indent=4)

    def _stringify_params(self, params):
        def stringify(v):
            if callable(v):
                return f"{v.__qualname__}"
            return v
        return {k: stringify(v) for k, v in params.items()}

    def _get_clean_params(self):
        return {
            "slic_params": self._stringify_params(self.slic_params),
            "matcher_params": self._stringify_params(self.matcher_params),
            "detector_params": self._stringify_params(self.detector_params),
        }

    def check_already_exists(self):
        current_params = self._get_clean_params()
        metadata = self._load_metadata()

        for entry in metadata:
            match = True
            for key in ["slic_params", "matcher_params", "detector_params"]:
                if entry.get(key) != current_params.get(key):
                    match = False
                    break
            if match:
                return entry["path"]
        return None

    def register_file(self, video_name):
        current_params = self._get_clean_params()
        entry = current_params.copy()
        entry["path"] = video_name
        entry["datetime"] = datetime.now().isoformat()

        metadata = self._load_metadata()
        metadata.append(entry)
        self._save_metadata(metadata)


    ## PLOTS ##
    def create_video(self, name=None,save_json=True, fps=7, plot_functions=None):
        if plot_functions is None:
            plot_functions=[
                lambda d, ax: d.show_overlay(ax=ax, unmatched_color=(1,0.7,0)),
                lambda d, ax: d.show_mask(ax=ax)
            ]

        os.makedirs(self.directory, exist_ok=True)

        metadata = self._load_metadata()
        if name is None:
            base = "video_"
            used_names = [entry.get("path", "") for entry in metadata]
            existing_nums = [
                int(re.search(rf"{base}(\d+)", path).group(1))
                for path in used_names if re.search(rf"{base}(\d+)", path)
            ]
            N = max(existing_nums, default=0) + 1
            name = f"{base}{N}"

        existing_name = self.check_already_exists()
        if existing_name is not None:
            existing_path = os.path.join(self.directory, existing_name)
            if os.path.exists(existing_path):
                print("Using cached video at:", existing_path)
                return existing_path
            raise Exception(f"File {existing_path} saved in metadata but cannot be found on disk")

        # Compute the frames if not done
        self.compute_all_frames()

        # Setup layout
        num_plots = len(plot_functions)
        cols = math.ceil(math.sqrt(num_plots))
        rows = math.ceil(num_plots / cols)
        figsize = (cols * 8, rows * 6)

        frames = []
        for i, detector in enumerate(tqdm(self.computed, desc="Processing Detectors")):
            fig, axs = plt.subplots(rows, cols, figsize=figsize)
            axs = np.array(axs).reshape(-1)
            fig.suptitle(f"Frame {i+1}", fontsize=16)

            for j, plot_func in enumerate(plot_functions):
                ax = axs[j]
                plot_func(detector, ax)

            # Hide any unused subplots
            for j in range(len(plot_functions), len(axs)):
                axs[j].axis("off")

            fig.canvas.draw()
            frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
            frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
            frames.append(frame)
            plt.close(fig)

            # Clear SP memory
            for SP in detector.matcher.SP_prev:
                SP.reset_info()

        video_name = f"{name}.mp4"
        video_path = os.path.join(self.directory, video_name)
        imageio.mimsave(
            video_path,
            frames,
            fps=fps,
            format='FFMPEG',
            codec='libx264',
            quality=10,
            pixelformat='yuv420p',
            macro_block_size=None
        )

        if save_json:self.register_file(video_name)
        print("Saved new video to:", video_path)
        return video_path

### DETECTOR WITH MEMORY

In [ ]:
class MemoryDetector(Detector):
    def __init__(self, matcher, prev_detector=None, remember_rate=0, **params):
        self.prev_detector = prev_detector
        self.remember_rate = remember_rate
        
        self.matcher = matcher
        self.tresh = params['tresh']
        self.dist_name = params['dist'].__name__
        self.detect(**params)


    def detect(self, dist, tresh, type="both", **params):
        self.prev_moved_ids = []
        self.curr_moved_ids = []

        #to plot the histogram
        self.real_distances = []

        self.prev_moved_mask = None
        self.curr_moved_mask = None

        #chooses which pairs of SP to use
        pairs = {
            "both": self.matcher.choices_both,
            "prev": self.matcher.choices_prev,
            "curr": self.matcher.choices_curr
        }.get(type)
        
        if pairs is None: raise Exception("Type of the pairs can only be 'both','prev' or 'curr'")

        # Initialize masks
        self.prev_moved_mask = np.zeros_like(self.matcher.prev.owner_map, dtype=bool)
        self.curr_moved_mask = np.zeros_like(self.matcher.curr.owner_map, dtype=bool)

        for sp_prev,sp_curr in pairs:
            #selct only some superpixels
            distance = dist(sp_prev,sp_curr)
            self.real_distances.append(distance)

            # obtain the moved distance from the previous detector
            prev_dist = distance
            if self.prev_detector is not None:
                for idx,prev_distance,_ in self.prev_detector.curr_moved_ids:
                    if sp_prev.idx == idx:
                        prev_dist = prev_distance #(np.exp(prev_distance -tresh) - 1) * np.abs(prev_dist - tresh)
                        break
                    
            #interpolate them 
            distance = (1-self.remember_rate) *distance + self.remember_rate * prev_dist 

            good = (distance >= tresh)            
            self.prev_moved_ids.append((sp_prev.idx,distance,good))
            self.curr_moved_ids.append((sp_curr.idx,distance,good))
            if good:                
                #adding to the masks the superpixel's region
                self.prev_moved_mask |= self.matcher.prev.owner_map == sp_prev.idx
                self.curr_moved_mask |= self.matcher.curr.owner_map == sp_curr.idx

            sp_prev.reset_info()

    def plot_distances_hist(self, bins=50, ax=None):
        real_distances = self.real_distances
        eval_distances = [d for _, d, _ in self.curr_moved_ids]

        if ax is None:
            fig, ax = plt.subplots(figsize=(6, 4))

        ax.hist(real_distances, bins=bins, alpha=0.6, color="steelblue", edgecolor="black", label="Real distances")
        ax.hist(eval_distances, bins=bins, alpha=0.6, color="green", edgecolor="black", label="Adjusted distances")

        ax.set_xlabel(f"DISTANCE.{self.dist_name}(SP)")
        ax.set_ylabel("Frequency")
        ax.set_title("Histogram of SP's changes")
        ax.grid(True)
        ax.axvline(self.tresh, color='red', linestyle='--', label=f'Threshold = {self.tresh:.2f}')
        ax.legend()


class MemoryFrameCollection(FrameCollection):
    @property
    def directory(self):
        return os.path.join("video_exports/interpolated/", self.subject)
    
    def copy_detectors(self, old, **detector_params):
        prev_detector = self.computed[-1] if self.computed else None

        for i in tqdm(range(len(old.computed)), desc="Recomputing Detectors"):
            matcher = old.computed[i].matcher  # ⚡ REUSE existing matcher

            # Reset info of the matcher.prev
            for SP in matcher.SP_prev:
                SP.reset_info()

            # Create a new MemoryDetector
            detector = MemoryDetector(matcher, prev_detector=prev_detector, **detector_params)

            self.computed.append(detector)
            prev_detector = detector
            
    def compute_frame(self):
        curr_frame = self._FrameCollection__get_new_frame()
        slic_prev = (
            SLIC(self.prev_frame, **self.slic_params)
            if not self.computed else self.computed[-1].matcher.curr
        )
        slic_curr = SLIC(curr_frame, **self.slic_params)
        matcher = Matcher(slic_prev, slic_curr, **self.matcher_params)

        prev_detector = self.computed[-1] if self.computed else None
        detector = MemoryDetector(matcher, prev_detector=prev_detector, **self.detector_params)
        
        for SP in detector.matcher.SP_prev:
            SP.reset_info()
            
        self.computed.append(detector)
        self.prev_frame = curr_frame
        return detector

# EXECUTION

The pipeline is divided into three parts, each with its own set of parameters:
- Segmentation
- Matching
- Detection

Let's examine each part in detail, highlighting the main challenges.  
To make observations easier, we will initially use the same sequence of frames, and later generalize to a broader set of examples.


In [ ]:
subway_files = list_files("images/video")
img1 = FrameCollection.load_image(subway_files[0])
img2 = FrameCollection.load_image(subway_files[1])
img10= FrameCollection.load_image(subway_files[9])

## SEGMENTATION

The first step is to compute an unsupervised segmentation to divide the image into regions.<br>
Initially, I created a class called `SLIC_old`, which is a direct implementation of the SLIC algorithm.
In particular, it enhances the provided SLIC implementation from the given notebook:
- By using static allocation for the distance matrix, it avoids unnecessary memory allocations.
- It precomputes a set of `distance_masks` at runtime: each instance of the object uses one of them (directly determined by its S and M values), and during the `assignment` step, it avoids recomputing the distance between the centroid and all the pixels in the search window.
- It enforces a stricter adherence to the original paper, correcting the search window size from **2S×2S** (as in the given code) to **S×S**.

These changes alone allow for a speed-up of up to 300%.


In [ ]:
S_values = [5, 15]
M_values = [5, 20]

fig, axes = plt.subplots(2, 2, figsize=(6, 6))

for i, S in enumerate(S_values):
    for j, M in enumerate(M_values):
        ax = axes[i, j]
        dist_mask = SLIC_old.init_distance_mask(M, S)
        im = ax.imshow(dist_mask, cmap='viridis', vmax=25)
        ax.set_title(f'S={S}, M={M}')
        ax.axis('off')
        fig.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle("Distance Masks for Different S and M", fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

However, since `skimage.segmentation.slic` is extremely fast, we will use it for behavioral observations.

To allow interchangeability between the two implementations, we define a wrapper class `SLIC`. 
(Of course, due to the differences in the underlying implementations, they will require different sets of parameters.)


A segmentation is considered "good" if it produces segments that:
1. Are small enough to isolate the moving object from the background, but large enough to not **over-segment** it.
2. Are **smooth** enough to not be sensitive to noise, but not so smooth that they ignore slow-moving objects.


For the segmentation part, we need to set three parameters:
1. `num_segments`: the number of **superpixels** we want to obtain.
2. `M`: the **compactness** of each region; the higher the value, the more squared the regions become.
3. `smooth`: the sigma of the low-pass filter applied before segmentation, used to reduce **noise**.


### 1) Choice of ```Num_segments```

In [ ]:
M = 30
s1 = SLIC(img1, num_segments=100, M=M)
s2 = SLIC(img1, num_segments=200, M=M)
s3 = SLIC(img1, num_segments=800, M=M)

fig, axes = plt.subplots(2, 3, figsize=(6*3, 4*2))

axes[0, 0].set_title("A) Too Few")
axes[0, 1].set_title("B) Good Enough")
axes[0, 2].set_title("C) Too Many")

s1.show_boundaries(c_idx=True, ax=axes[0, 0])      
s2.show_boundaries(centroids=True, ax=axes[0, 1])  
s3.show_boundaries(centroids=True, ax=axes[0, 2]) 

s1.explained_color_variance(ax=axes[1, 0])
s2.explained_color_variance(ax=axes[1, 1])
s3.explained_color_variance(ax=axes[1, 2])
plt.tight_layout()
plt.show()

As we can see, the choice of **M** is not straightforward.<br>
A helpful metric could be the **Explained Color Variance**: our goal is to minimize it, i.e., the variance within each Superpixel in the color space.<br>
Obviously, this is not the only metric—otherwise, the best solution would always be to build each Superpixel with only 1 pixel (variance = 0), which is not practical.

As mentioned earlier, we can understand why some segmentations are better than others:
- **A)** has regions that are too large, such as 73 (that both contains the luggage and the person's legs): the **detection** will likely be inaccurate, and the **matching** will hence be less reliable.
- **C)** has regions that are too small: although the **detection** would likely be more accurate, the **matching** would become much more difficult, since moving objects may not have the same number of segments across consecutive frames (as shown in the image below): there will be for sure some mismatch.


In [ ]:
Matcher.plot_overlay(s3,SLIC(img2, num_segments=800, M=M),img_bg = img2)

Another observation we can make is how sensitive the segmentation can be to noise, as we can see in the top part of the image.

Again, a **instable segmentation** will produce difficult (impossible) matching.

### 2) Choice of ```M```

In [ ]:
num_segments = 200
s4 = SLIC(img1, num_segments=num_segments, M=10)
s5 = SLIC(img1, num_segments=num_segments, M=50)
s6 = SLIC(img1, num_segments=num_segments, M=70)

fig, axes = plt.subplots(2, 3, figsize=(6*3, 4*2))

axes[0, 0].set_title("A) Too \"fuzzy\"")
axes[0, 1].set_title("B) Good Enough")
axes[0, 2].set_title("C) Too \"rigid\"")

s4.show_boundaries(centroids=True, ax=axes[0, 0])
s5.show_boundaries(centroids=True, ax=axes[0, 1])
s6.show_boundaries(centroids=True, ax=axes[0, 2])

Matcher.plot_overlay(s4,SLIC(img2, num_segments=num_segments, M=10), centroids=True, img_bg = img2, ax=axes[1, 0])
Matcher.plot_overlay(s5,SLIC(img2, num_segments=num_segments, M=50), centroids=True, img_bg = img2, ax=axes[1, 1])
Matcher.plot_overlay(s6,SLIC(img2, num_segments=num_segments, M=70), centroids=True, img_bg = img2, ax=axes[1, 2])

plt.tight_layout()
plt.show()

As predicted, the choice of M has a huge impact on the quality of the segmentation:
- A) is extremely **noise-dependent**: even very small changes from one frame to the next can produce extremely different **Superpixels**.
- C) for the opposite reason, suffers from the same problem: during the iterative process of the SLIC, a displacement of the centroid causes irreversible changes in the segmentation.
- B) seems to be the best choice, but again is not perfect: in the top-left part, we can see some noisy segments. With smoothing, we can further reduce this problem.


### 3) Choice of ```smooth```

In [ ]:
M = 50
s7 = SLIC(img1, num_segments=num_segments, M=M, smooth=None)
s8 = SLIC(img1, num_segments=num_segments, M=M, smooth=1)
s9 = SLIC(img1, num_segments=num_segments, M=M, smooth=5)

fig, axes = plt.subplots(2, 3, figsize=(6*3, 4*2))

axes[0, 0].set_title("A) Too \"fuzzy\"")
axes[0, 1].set_title("B) Good Enough")
axes[0, 2].set_title("C) Too \"smoth\"")

s7.show_boundaries(ax=axes[0, 0],img = "data")
s8.show_boundaries(ax=axes[0, 1],img = "data")
s9.show_boundaries(ax=axes[0, 2],img = "data")

Matcher.plot_overlay(s7,SLIC(img2, num_segments=num_segments, M=M, smooth=None), img_bg = lab2rgb(s7.data), ax=axes[1, 0])
Matcher.plot_overlay(s8,SLIC(img2, num_segments=num_segments, M=M, smooth=1   ), img_bg = lab2rgb(s8.data), ax=axes[1, 1])
Matcher.plot_overlay(s9,SLIC(img2, num_segments=num_segments, M=M, smooth=5.0 ), img_bg = lab2rgb(s9.data), ax=axes[1, 2])

plt.tight_layout()
plt.show()

It is immediately clear how much this pre-processing impacts the results:
- A) With no smoothing, on the sop part there is some noise.
- B) there is an improvement, but it still shows some **noise-dependent Superpixels** in the top part.
- C) the issues at the top are resolved, but the ```SLIC``` completely loses its ability to recognize (and thus detect motion in) smaller objects, or objects with a color too similar to the background, like we show below.

### SLIC result

In [ ]:
subway_process = FrameCollection.create(subway_files,subject="subway",
    slic_params= {
        "M": 50,
        "num_segments": 200,
        "smooth": 1
    },
    matcher_params= {"kernel" : KERNEL.xy}, #placeholders
    detector_params= {"dist": DISTANCE.xy,"tresh":0}  #placeholders
)
output_video = subway_process.create_video(name="subway_slic",
    plot_functions=[
        lambda d, ax: d.matcher.curr.show_boundaries(centroids=True,legend=False,ax=ax)
    ],
    fps=7)
Video(output_video, embed=True)

Again, it still has some issues, but for now we'll keep those parameters and continue with our pipeline's tinkering.

In [ ]:
#gragabe collection
del s1,s2,s3,s4,s5,s6,s7,s8,s9
gc.collect()

slic_1 =  SLIC(img1,  num_segments=200, M=50, smooth=1)
slic_2 =  SLIC(img2,  num_segments=200, M=50, smooth=1)
slic_10 = SLIC(img10, num_segments=200, M=50, smooth=1)

## MATCHING

The matching procedure aims to create continuity in the segmentation along the **time axis**.<br>
First, it describes each ```Superpixel``` in both frames and then builds the ```similarity_matrix``` to create a bijective map between them.<br>

The tuning in this step consists of choosing a ```KERNEL``` that defines a similarity function, based on a ```DISTANCE``` in a space of choice.

By selecting the appropriate space and tuning the related parameters, we can effectively **match** the Superpixels across different frames.


### 1) Centroid's Position

The immediate idea is associate each SP with the one with the closest **centroid**.<br>

Let's visualize how well this kernel performs.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6),gridspec_kw={'width_ratios': [1, 2]})  # or any size that fits
fig.suptitle("Matching using the Centroid's Position", fontsize=20)

m1 = Matcher(slic_1,slic_2,
    kernel = KERNEL.xy,
    s_xy = 30
)

m1.plot_matrix(ax=ax1)
m1.show_match(boundaries=True, tresh_xy=4, c_unmatched=(1,0.7,0), ax=ax2)

plt.show()

It seems a very good way to match superpixels; but how well does it performs, it the displacement is **large** (i.e. the object moves very fast)?

To simulate this, we simply increment the delta t.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1, 2]})
fig.suptitle("Matching using the Centroid's Position", fontsize=20)

m1_b = Matcher(slic_1,slic_10,
    kernel = KERNEL.xy,
    s_xy = 30
)

m1_b.show_overlay(ax1, centroids=True, img_bg=m1_b.curr.original)
m1_b.show_match(boundaries=True, tresh_xy=7, c_unmatched=(1,0.7,0), ax=ax2)

plt.show()

As we can clearly see, we have a clear mismatch on the moving superpixels.<br>
Also, some of them (in this case, the suitcase), could even not have a respective counterpart: this problem is not always solvable, and we have to try to **limitate his effect** on the full pipeline.

In this case we have **artificially** created the problem, but in the examples below we can see that is a real criticality.

### 2) Superpixel Description

An alternative approach is to describe each Superpixel under the **assumption** (not always true) that the moving object's **Superpixels maintain the same structure**, and only change their position.

Following this idea, different kernels are proposed. To better illustrate their effectiveness, we will use another set of images where the **movement** is much larger.


In [ ]:
tennis_files = list_files("images/stennis")
slic_ten_1 = SLIC(FrameCollection.load_image(tennis_files[0]), num_segments=300, M=30,smooth=1)
slic_ten_2 = SLIC(FrameCollection.load_image(tennis_files[1]), num_segments=300, M=30,smooth=1)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1, 2]})
fig.suptitle("Matching using the Centroid's Position", fontsize=20)


m1_b = Matcher(slic_ten_1,slic_ten_2,
    kernel = KERNEL.xy,
    s_xy = 30
)

m1_b.plot_matrix(ax=ax1)
m1_b.show_match(boundaries=True, idx_pair=[196,197,198,71],c_unmatched=(1,0.7,0), ax=ax2)

plt.show()

With this images is immediate the need of a new `KERNEL`, since the that one is not working so well.

#### A) RGB Space

The descriptor chosen is the mean of the pixels within the ```Superpixel```, computed in the **RGB** space.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6),gridspec_kw={'width_ratios': [1, 2]})  # or any size that fits
fig.suptitle("Matching using the Mean RGB value", fontsize=20)

m2 = Matcher(slic_ten_1,slic_ten_2,
    kernel = KERNEL.rgb,
    s_rgb = 0.1
)
m2.plot_matrix(ax=ax1)
m2.show_match(boundaries=True, tresh_xy=30,c_unmatched=(1,0.7,0), ax=ax2)

plt.show()

Obviously, this kernel must operate in a local space, so it has to be integrated with the ```KERNEL.xy```.<br>
Since we are trying to define a **distance** in an inhomogeneous space, we need to adjust the **variances** of the kernels to combine them properly (the positional variance is naturally much larger than the variance in the RGB space, which is normalized between 0 and 1; thus, proper scaling is required to balance their influence).


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6),gridspec_kw={'width_ratios': [1, 2]})  # or any size that fits
fig.suptitle("Matching using the Mean RGB value and Centroid's position", fontsize=20)

m2 = Matcher(slic_ten_1,slic_ten_2,
    kernel = KERNEL.rgb_xy,
    alpha = 0.6,
    s_rgb = 0.3,
    s_xy = 30
)
m2.plot_matrix(ax=ax1)
m2.show_match(boundaries=True, tresh_xy=15,c_unmatched=(1,0.7,0), ax=ax2)

Seems like it should work, so let's empirically visualize if there are some obvoius mismatch.

To do it, a pretty effective way is to compute a very simple version of the `optical flow`, i.e. print the **displacement**'s vectors.

In [ ]:
tennis_process = FrameCollection.create(tennis_files,subject="tennis",
    slic_params= {
        "M": 30,
        "num_segments": 300,
        "smooth": 1
    },
    matcher_params= {
        "kernel" : KERNEL.rgb_xy,
        "alpha" : 0.6,
        "s_rgb" : 0.3,
        "s_xy" : 30
    },
    detector_params= {"dist": DISTANCE.xy,"tresh": 10}
)

output_video = tennis_process.create_video(name="tennis_matching",
    plot_functions=[
        lambda d, ax: d.matcher.show_match_v2(type="both",boundaries=False, tresh_xy=5,c_unmatched=(1,0.7,0),ax=ax)
    ],
    fps=2)
Video(output_video, embed=True)

The result is acceptable, but for academic purposes we will try also some others `KERNELS`, to make some more observations.

#### B) Patch around the center

Given the centroid, we use a patch around it, just like we do to match corners.

We can already predict that this idea will not be very effective, since the underlying idea is that the information of the **corner's surroundings** is enough to uniquely define them.<br>
However, since centroids are usually located in flat regions (by construction), they are much more similar to a **blob** than a corner, hence the same observations hold.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6),gridspec_kw={'width_ratios': [1, 2]}) 
fig.suptitle("Matching using the Patch around the centroid", fontsize=20)

m3 = Matcher(slic_ten_1,slic_ten_2,
    kernel = KERNEL.patch,
    s_patch = 5
)
m3.plot_matrix(ax=ax1)
m3.show_match(type="both",boundaries=True, tresh_xy=4,c_unmatched=(1,0.7,0), ax=ax2)

plt.show()

As predicted, we have a lot of errors; in particular, there are not so many mismatches, but a lot of superpixels are unmatched.

Another way to highlight the inefficiency of this matching approach is to visualize how many times each superpixel is "chosen" (i.e., maximizes **similarity**) by those in the other frame.

Since our goal is a "theoretical" **bijective** map, any superpixel that has been chosen a number of times different from 1 is an indication of error.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6)) 

m3.show_choices(type="prev",boundaries=(1,1,1),ax=ax1,text_size={ "start": 3,"step": 3 })
m3.show_choices(type="curr",boundaries=(1,1,1),ax=ax2,text_size={ "start": 3,"step": 3 })

plt.show()

As before, let's see if by using the **Centroid's position** the performance improves.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6),gridspec_kw={'width_ratios': [1, 2]})  # or any size that fits
fig.suptitle("Matching using the centroid's Patch and position", fontsize=20)

m3 = Matcher(slic_ten_1,slic_ten_2,
    kernel = KERNEL.patch_xy,
    s_patch = 5,
    s_xy = 30
)
m3.plot_matrix(ax=ax1)
m3.show_match(type="both",boundaries=True, tresh_xy=5,c_unmatched=(1,0.7,0), ax=ax2)

plt.show()

Despite this modification, we still observe many superpixels that are not **matched** (the orange ones).

In [ ]:
output_video = tennis_process.copy(
    matcher_params= {
        "kernel" : KERNEL.patch_xy,
        "s_patch" : 5,
        "s_xy" : 30
    }
).create_video(name="tennis_matching_2",
    plot_functions=[
        lambda d, ax: d.matcher.show_match_v2(type="both",boundaries=False, tresh_xy=5,c_unmatched=(1,0.7,0),ax=ax)
    ],
    fps=2)
Video(output_video, embed=True,width=800)

Another limitation of this approach is related to the geometry of superpixels: since we cannot assume that the centroids (and therefore the patches) lie within the superpixel itself (convexity is not guaranteed), the method proves to be intrinsically unreliable.

For this reason, we prefer the previous version, `KERNEL.rgb_xy`.  


In [ ]:
#grabage collecting
del m1,m1_b,m2,m3,slic_ten_1,slic_ten_2
gc.collect()

matcher_1_2 = Matcher(slic_1,slic_2,kernel = KERNEL.xy)

## MOTION DETECTION

The final step involves evaluating a metric for each pair of superpixels to determine if it can be considered as having "moved."

Once again, we can choose the appropriate metric by calling the methods of the `DISTANCE` class.


### 1) Centroid's displacement

We simply evaluate how much each ```Centroid``` has moved: if is greater than a value, it is considered a **movement** (and not "noise").

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

det1 = Detector(matcher_1_2,
    dist=DISTANCE.xy,
    tresh = 4
)

det1.show_overlay(ax=ax1,show_dist=True)
det1.plot_distances_hist(ax=ax2)

plt.tight_layout()
plt.show()

Seems a pretty effective way to detect motion: let's visualize it in action. 

In [ ]:
output_video = subway_process.copy(
    matcher_params={
        "kernel":KERNEL.xy
    },
    detector_params={
        "dist" :DISTANCE.xy,
        "tresh":4
    }
).create_video()
Video(output_video, embed=True)

As we can observe, the **metric** used for this purpose is not very efficient:
- The girl in the upper part of the video is **detected** only briefly when she transitions from one Superpixel to another: the Superpixels change enough only when she crosses the boundary (as previously observed during the `segmentation` procedure).
- The men in the foreground are almost always detected, although sometimes some Superpixels lose their previous correspondence.
- Moving objects not only cause **displacements** in their respective Superpixels but also in the surrounding areas.

Another key limitation of this approach is that when a moving object crosses a **Superpixel boundary**, it briefly creates a new Superpixel without matching it to the previous one. This further undermines the stability of the solution.

For now, we will try to address this issue by modifying the detection metric.


### 2) IOU

Since the centroid displacement can be caused by **random noise or small changes**, we attempt to use a metric that works across the entire Superpixel: by overlapping the two frames, we apply the `INTERSECTION OVER UNION (IOU)` metric.

If this value falls below a threshold, it indicates a significant change, hence motion.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

det2 = Detector(matcher_1_2,
    dist=DISTANCE.iou,
    tresh = -0.8
)

det2.show_overlay(ax=ax1,show_dist=True)
det2.plot_distances_hist(ax=ax2)

plt.tight_layout()
plt.show()

In [ ]:
output_video = subway_process.copy(
    detector_params={
        "dist" :DISTANCE.iou,
        "tresh":-0.8
    }
).create_video()
Video(output_video, embed=True)


It seems that this approach could be quite effective, but only if there is essentially **no noise**, requiring a very strict segmentation; otherwise, there will be many **false positives** in the detection.<br>
This issue could be mitigated by considering the *time dimension* as well, introducing an idea of continuity (we will explore this in a simple version at the end).

In particular, we observe that the Superpixels (SPs) that are actually moving maintain their **shape** invariant, while the **false positives** around them are only deformed, often by "compression" on one side.<br>
Let’s try to leverage this observation to build a more robust `Detector`.


### 3) Overlapping Threshold

We aim to use a metric where a superpixel is considered to have moved only if, when comparing the masks of SP_t-1 and SP_t (denoted as a and b, respectively), the areas of `a\b` and `b\a` (normalized) both exceed a threshold.

This changes greatly diminuish the **false positives** around the objects (but still is not prefect).


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

det3 = Detector(matcher_1_2,
    dist=DISTANCE.overlap,
    tresh = 12
)

det3.show_overlay(ax=ax1,show_dist=True)
det3.plot_distances_hist(ax=ax2)

plt.tight_layout()
plt.show()

In [ ]:
output_video = subway_process.copy(
    detector_params={
        "dist" :DISTANCE.overlap,
        "tresh": 12
    }
).create_video()
Video(output_video, embed=True)

## OPTIMIZATION

The execution pipeline, as described previously, seems to have reached a point where further improvements cannot be made. We now introduce an assumption, directly derived from an empirical observation of the videos used as examples: **superpixels in motion in one frame tend to maintain roughly the same movement in the following frame**.

This approach allows us to reduce false positives, that is, superpixels that "appear" in the detector's mask for just one frame.<br>
Of course, this method requires a particularly robust matching procedure, as a mismatch could introduce artifacts in the detection.

To address this, we use the `MemoryDetector` class, which requires a `remember_rate` parameter. This parameter allows us to compute the distance between two matched superpixels as:<br>

```(1 - remember_rate) * curr_dist + remember_rate * prev_dist```

where `prev_dist` is the distance computed by the detector in the previous frame.


In [ ]:
output_video = subway_process.copy(
    detector_params={
        "dist" :DISTANCE.overlap,
        "tresh": 12,
        "remember_rate":0.75
    }
).create_video()
Video(output_video, embed=True)

We can consider this approach as the *state of the art* within this notebook, as the **limitations** discussed prevent further improvements in the results.

In these results, we deliberately avoid considering the **unmatched SuperPixels**, as managing them is too specific to each example and does not provide significant improvements: we simply consider them as **moving**.

One alternative we could explore is to start from scratch, such as increasing the number of superpixels in the SLIC segmentation, in an attempt to detect smaller (and slower) objects, like the girl in the background.<br>
While we already know this is a challenging task, we will demonstrate some other attempts to address it.


# FINAL RESULTS

In [ ]:
output_video = subway_process.copy(
    slic_params= {
        "M": 50,
        "num_segments": 200,
        "smooth": 1
    },
    matcher_params= {
        "kernel" : KERNEL.xy
    },
    detector_params={
        "dist" :DISTANCE.overlap,
        "tresh": 12,
        "remember_rate":0.7
    }
).create_video(
    name="integral_subway",
    plot_functions=[
        lambda d, ax: d.matcher.curr.show_boundaries(centroids=True,ax=ax),
        lambda d, ax: d.matcher.show_match_v2(boundaries=False,tresh_xy=6,c_unmatched=(1,0.7,0),ax=ax),
        lambda d, ax: d.show_overlay(ax=ax,unmatched_color=(1,0.7,0)),
        lambda d, ax: d.show_mask(ax=ax)
    ],
    fps=9
)
Video(output_video, embed=True)

In [ ]:
output_video = tennis_process.copy(
    slic_params= {
        "M": 30,
        "num_segments": 300,
        "smooth":2
    },
    matcher_params= {
        "kernel" : KERNEL.rgb_xy,
        "alpha" : 0.5,
        "s_rgb" : 5,
        "s_xy" : 30
    },
    detector_params= {
        "dist": DISTANCE.overlap,
        "tresh": 17,
        "remember_rate":0.1
    }
).create_video(
    name="integral_tennis",
    plot_functions=[
        lambda d, ax: d.matcher.curr.show_boundaries(centroids=True,ax=ax),
        lambda d, ax: d.matcher.show_match_v2(boundaries=False,tresh_xy=3,c_unmatched=(1,0.7,0),ax=ax),
        lambda d, ax: d.show_overlay(ax=ax,unmatched_color=(1,0.7,0)),
        lambda d, ax: d.show_mask(ax=ax)
    ],
    fps=5
)
Video(output_video, embed=True)

In [ ]:
output_video = FrameCollection.create(list_files("images/peoples"),subject="peoples",
    slic_params= {
        "M": 40,
        "num_segments": 1000,
        "smooth":2
    },
    matcher_params= {
        "kernel" : KERNEL.xy
    },
    detector_params= {
        "dist": DISTANCE.overlap,
        "tresh": 3.5,
        "remember_rate":0.25
    }
).create_video(
    name="integral_peoples",
    plot_functions=[
        lambda d, ax: d.matcher.curr.show_boundaries(centroids=True,ax=ax),
        lambda d, ax: d.matcher.show_match_v2(boundaries=False,tresh_xy=2,c_unmatched=(1,0.7,0),ax=ax),
        lambda d, ax: d.show_overlay(ax=ax,unmatched_color=(1,0.7,0)),
        lambda d, ax: d.show_mask(ax=ax)
    ],
    fps=10
)
Video(output_video, embed=True)

In [ ]:
output_video = FrameCollection.create(list_files("./images/dancer"),subject="dancer",
    slic_params= {
        "M": 35,
        "num_segments": 150,
        "smooth":3
    },
    matcher_params= {
        "kernel" : KERNEL.xy
    },
    detector_params= {
        "dist": DISTANCE.overlap,
        "tresh": 8,
        "remember_rate":0.5  
    }
).create_video(
    name="integral_dancer",
    plot_functions=[
        lambda d, ax: d.matcher.curr.show_boundaries(centroids=True,ax=ax),
        lambda d, ax: d.matcher.show_match_v2(boundaries=False,tresh_xy=4,c_unmatched=(1,0.7,0),ax=ax),
        lambda d, ax: d.show_overlay(ax=ax,unmatched_color=(1,0.7,0)),
        lambda d, ax: d.show_mask(ax=ax)
    ],
    fps=15
)
Video(output_video, embed=True)